# UC CosMx: Validation 1K Processing and Annotation Comparison (Fig 2L, Ext Fig 6H)

Serial-cut 1K validation experiment:
1. QC and normalization
2. Annotation comparison between 1K-only and imputed 1K
3. Marker dotplots

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/uc_serial_cuts/1000plex"
OUTPUT_DIR = "/path/to/cosmx_data/uc_serial_cuts/1000plex/Processed_merged"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import osfrom scipy.sparse import csr_matrix
import anndata
import scipy.stats as stats


In [ ]:
sample_dir = '/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/flat'

In [ ]:
counts = pd.read_csv(sample_dir+'/UC_val_dir_exprMat_file.csv', index_col=0)          # Rows = genes, Cols = cells


In [ ]:
fov_positions = pd.read_csv(sample_dir+'/UC_val_dir_fov_positions_file.csv')  # Contains x, y


In [ ]:
metadata = pd.read_csv(sample_dir+'/UC_val_dir_metadata_file.csv')      # Indexed by cell_id

In [ ]:
counts_clean = counts.drop(['cell_ID'],axis = 1)

In [ ]:
counts_clean.index = counts_clean['cell']
counts_clean = counts_clean.drop(['cell'],axis = 1)
counts_clean

In [ ]:
# Create AnnData object
adata = sc.AnnData(X=counts_clean)

In [ ]:
# Add metadata to AnnData
adata.obs = metadata
adata

In [ ]:
adata.var['gene'] = adata.var_names.tolist()

In [ ]:
adata.obsm['spatial_fov']=adata.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_probe']=[True if 'SystemControl' in i else False for i in adata.var.index.tolist()]
adata.var

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]]
adata=adata[:, ~adata.var["control_probe"]]

In [ ]:

if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)


In [ ]:

sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)

In [ ]:

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=50)
sc.pp.filter_cells(adata, min_genes=25)
sc.pp.filter_cells(adata, max_counts=800)
sc.pp.filter_cells(adata, max_genes=350)
adata=adata[adata.obs['Area']<=30000]

In [ ]:
adata.obs[['orig.ident']]='UC_V2_UC_V1'
adata.obs['patient']=['UC_V1' if int(i) <=28 else 'UC_V2' for i in adata.obs['fov'].tolist() ]
adata.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata.obs['fov'].tolist(), adata.obs['cell_ID'].tolist())]


In [ ]:
slide=[]
for i,j in zip(adata.obs['orig.ident'].tolist(),adata.obs['fov'].tolist()):
    j= int(j)
    if j <=12:
        slide.append(i+'_'+'slide1')
    elif j > 12 and j <= 28:
        slide.append(i+'_'+'slide2')
    elif j > 28 and j <= 39:
        slide.append(i+'_'+'slide3')
    else:
        slide.append(i+'_'+'slide4')

In [ ]:
adata.obs['slide']=slide

In [ ]:
adata.layers['raw']=pd.DataFrame(adata.X.toarray())

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)


In [ ]:
adata.write_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')


In [ ]:
adata_norm_nolog1p = adata.copy()
adata_norm_nolog1p.X = adata.layers['raw']

In [ ]:
sc.pp.normalize_total(adata_norm_nolog1p, inplace=True)

In [ ]:
adata_norm_nolog1p_count = pd.DataFrame(adata_norm_nolog1p.X)

In [ ]:
adata_norm_nolog1p_count.columns = adata_norm_nolog1p.var.index.tolist()
adata_norm_nolog1p_count.index=adata_norm_nolog1p.obs['fov_cell_id'].tolist()
adata_norm_nolog1p_count

In [ ]:
adata_norm_nolog1p_count.to_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/count_csv/norm_counts.csv')

# anno

In [ ]:
adata=sc.read_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')


In [ ]:
noimp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/anno/norm_counts_aucell.csv')

In [ ]:
imp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir//anno/imp_log1p_counts_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')

In [ ]:
imp_anno=imp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')
noimp_anno=noimp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')

In [ ]:
adata.obs['v7_ywl_b_imp_cell_type_short'] = imp_anno['cell_type_short'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_full'] = imp_anno['cell_type'].tolist()
adata.obs['v7_ywl_b_imp_cell_category_from_type'] = imp_anno['cell_category'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_general'] = imp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['cell_type_short'] = noimp_anno['cell_type_short'].tolist()
adata.obs['cell_type_full'] = noimp_anno['cell_type'].tolist()
adata.obs['cell_category_from_type'] = noimp_anno['cell_category'].tolist()
adata.obs['cell_type_general'] = noimp_anno['cell_type_general'].tolist()

In [ ]:
# Create a color palette
coordinates = adata.obsm['spatial_fov']

category_column = 'cell_category_from_type'
categories = adata.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("No Imp Cell Types")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
# Create a color palette
coordinates = adata.obsm['spatial_fov']

category_column = 'v7_ywl_b_imp_cell_category_from_type'
categories = adata.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("YWL Imp Cell Types")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
noimp_mye = adata[adata.obs['cell_category_from_type']=='Myeloid']
imp_mye = adata[adata.obs['v7_ywl_b_imp_cell_category_from_type']=='Myeloid']

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['myeloid_binary_v7_ywl_b_imp_cell_category_from_type'] = [1 if i == 'Myeloid' else 0 for i in adata.obs['v7_ywl_b_imp_cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.CD68'], adata.obs['myeloid_binary_v7_ywl_b_imp_cell_category_from_type'])
spearman['CD68'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['myeloid_binary_cell_category_from_type'] = [1 if i == 'Myeloid' else 0 for i in adata.obs['cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.CD68'], adata.obs['myeloid_binary_cell_category_from_type'])
spearman['CD68'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['epi_binary_v7_ywl_b_imp_cell_category_from_type'] = [1 if i == 'Epithelial' else 0 for i in adata.obs['v7_ywl_b_imp_cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.PanCK'], adata.obs['epi_binary_v7_ywl_b_imp_cell_category_from_type'])
spearman['PanCK'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['epi_binary_cell_category_from_type'] = [1 if i == 'Epithelial' else 0 for i in adata.obs['cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.PanCK'], adata.obs['epi_binary_cell_category_from_type'])
spearman['PanCK'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['imm_binary_v7_ywl_b_imp_cell_category_from_type'] = [1 if i in (['Myeloid','Granulocyte','T','B']) else 0 for i in adata.obs['v7_ywl_b_imp_cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.CD45'], adata.obs['imm_binary_v7_ywl_b_imp_cell_category_from_type'])
spearman['CD45'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:

# Create a binary column for 'Myeloid' presence
adata.obs['imm_binary_cell_category_from_type'] = [1 if i in ['Myeloid','Granulocyte','T','B'] else 0 for i in adata.obs['cell_category_from_type'].tolist() ]

# Calculate Spearman's correlation between Max.CD68 and 'Myeloid' presence
spearman_corr, p_value = stats.spearmanr(adata.obs['Max.CD45'], adata.obs['imm_binary_cell_category_from_type'])
spearman['CD45'].append(spearman_corr)
print(f'Spearman correlation: {spearman_corr}, p-value: {p_value}')

In [ ]:
spearman_df = pd.DataFrame(spearman).T

In [ ]:
spearman_df.columns = ['Imputed','No Imputation']

# read in supporting data

In [ ]:
wtx=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

In [ ]:
yawei_imp = pd.read_csv('/path/to/imputation_model/results/Jenny_cosmx_geneimputation/results/1K_imputation/external_validation/1k_pred_b_log1p.csv')

In [ ]:
yawei_imp.index = yawei_imp['old_index'].tolist()
yawei_imp=yawei_imp.drop('old_index',axis = 1)
yawei_imp.index = [str(i) for i in yawei_imp.index.tolist()]
yawei_imp

In [ ]:
ym_df=pd.DataFrame(adata.X.toarray())
ym_df.columns = adata.var_names.tolist()
ym_df.index = adata.obs_names.tolist()
ym_df

In [ ]:
merge1k = ym_df.merge(yawei_imp, left_index=True, right_index=True)

In [ ]:
merge1k.columns.unique()

In [ ]:
adata_imp = anndata.AnnData(X=merge1k.values) 
adata_imp.obs_names = merge1k.index  # Cell names
adata_imp.var_names = merge1k.columns  # Gene names
adata_imp.obs = adata.obs
adata_imp.obs=adata_imp.obs.drop(['unassignedTranscripts', 'nCount_RNA', 'nFeature_RNA', 'nCount_negprobes', 'nFeature_negprobes', 'nCount_falsecode', 'nFeature_falsecode', 'Area.um2', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'cell', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'n_counts', 'n_genes'],axis = 1)

In [ ]:
adata_imp_mye = adata_imp[adata_imp.obs['v7_ywl_b_imp_cell_category_from_type']=='Myeloid']
adata_imp_mye

In [ ]:
overlapping_dc_markers = [
    "CPVL", "LGALS2", "DNASE1L3", "IDO1", "MYCL", "HLA-DPB1",
    "HLA-DQB2", "P2RY6", "SLAMF8", "HLA-DPA1",
    "CSF2RA", "HLA-DRB5", "HLA-DQA1", "HLA-DRB1", "HLA-DQB1",
    "HLA-DRA", "SPI1", "HLA-DQA2", "HCK"
]
dc_markers = []
for i in overlapping_dc_markers:
    if i in adata.var_names.tolist():
        dc_markers.append(i)

In [ ]:
overlapping_momac_markers = [
    "RETN", "FCN1", "CD300E", "EREG", "LILRA1", "LILRA5", "OLR1",
    "APOBEC3A", "LILRB2", "CLEC4E", "CD163", "CALHM6", "LILRB1",
    "ADGRE2", "EPB41L3", "KYNU"
]
momac=[]
for i in overlapping_momac_markers:
    if i in adata.var_names.tolist():
        momac.append(i)
momac

In [ ]:
Neutrophil = ['FCGR3B', 'CMTM2', 'CXCR2', 'HCAR3', 'CXCR1', 'PROK2', 'FFAR2', 'ADGRG3', 'CSF3R', 'AL034397.3', 'FPR2', 'HCAR2', 'KCNJ15', 'CXCL8', 'SMIM25', 'AZIN1-AS1', 'LUCAT1', 'AQP9', 'S100A8', 'FPR1', 'TREM1', 'S100A12', 'CYP4F3', 'IL1RN', 'MIR3945HG', 'G0S2', 'S100A9', 'KCNJ2', 'LILRB3', 'AC015912.3', 'MNDA', 'DGAT2', 'AC023157.3', 'TLR4', 'PLEK', 'P2RY13', 'MMP25', 'ACSL1', 'MME', 'TLR2', 'IL1B', 'ALPL', 'LRRK2', 'NAMPT', 'STEAP4', 'NCF2', 'TNFAIP6', 'OSM', 'C5AR1', 'CLEC4E']
neut=[]
for i in Neutrophil:
    if i in adata.var_names.tolist():
        neut.append(i)
neut

In [ ]:
Cycl_Myeloid  =['CDK1', 'SPC25', 'UBE2C', 'RRM2', 'CEP55', 'AURKB', 'GTSE1', 'CENPA', 'TYMS', 'CCNA2', 'BIRC5', 'TOP2A', 'SHCBP1', 'UHRF1', 'TK1', 'CDCA3', 'RAD51AP1', 'CCNB2', 'CDCA5', 'HMMR', 'CDC20', 'C1QC', 'ZWINT', 'TPX2', 'C1QB', 'ASF1B', 'MKI67', 'CDKN3', 'VSIG4', 'C1QA', 'MS4A4A', 'FAM111B', 'NUSAP1', 'MS4A6A', 'PRC1', 'ADORA3', 'CLSPN', 'CCNB1', 'SDS', 'LYZ', 'DNASE1L3', 'CDT1', 'P2RY6', 'STMN1', 'AIF1', 'CENPF', 'UBE2T', 'CSF1R', 'ADAP2', 'MMP12']
cycl=[]
for i in Cycl_Myeloid:
    if i in adata.var_names.tolist():
        cycl.append(i)
cycl

In [ ]:
sel=['MS4A4A','MMP12','GPNMB','APOC1']

In [ ]:
mye_markers_1k=['UBE2C','TOP2A','HLA-DPA1','CSF2RA','IDO1','OLR1','ADGRE2','CLEC4E','CD163',
                'CXCR2','CXCR1','FFAR2','SELENOP','MMP9','SPP1','MMP12','GPNMB','APOC1']
markers = {'Cycl Myeloid': ['UBE2C','TOP2A'], 'DC': ['HLA-DPA1','CSF2RA'], 'Mo-mac': ['OLR1','ADGRE2','CLEC4E',],
          'Neurophil': ['CXCR2','CXCR1','FFAR2'],'SELENOP+ Mac': ['SELENOP','MMP9','MMP12','GPNMB','APOC1']}


In [ ]:
# if not analyzing the entire imputed matrix
#data_imp  = adata.copy()
adata_imp_mye = adata_imp[adata_imp.obs['v7_ywl_b_imp_cell_category_from_type']=='Myeloid']

In [ ]:
dp = sc.pl.dotplot(
    adata_imp_mye, mye_markers_1k, 'v7_ywl_b_imp_cell_type_general', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()


In [ ]:
dp = sc.pl.dotplot(
    adata_no_imp_mye, mye_markers_1k, 'cell_type_general', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()


# dot plot coarse

In [ ]:
Myeloid =['FPR2', 'HCAR3', 'AQP9', 'SMIM25', 'IL1RN', 'FPR1', 'S100A8', 'CSF3R', 'IL1B', 'S100A9', 'P2RY13', 'IGSF6', 'LILRB2', 'C5AR1', 'CD163', 'TREM1', 'CXCL8', 'MMP12', 'C1QC', 'C1QB', 'C1QA', 'VSIG4', 'CLEC10A', 'PLA2G7', 'FPR3', 'CSF1R', 'MS4A4A', 'MS4A6A', 'MNDA', 'FCGR2A', 'NCF2', 'CSTA', 'CPVL', 'TLR4', 'DNASE1L3', 'LILRB3', 'PILRA', 'LYZ', 'AIF1', 'MS4A7', 'CLEC7A', 'G0S2', 'CLEC4A', 'CSF2RA', 'STAB1', 'CD14', 'CCR1', 'ITGAX', 'ACSL1', 'HCK']
Connective = ['TFPI2', 'ADAMDEC1', 'CCL13', 'CCL11', 'PCDH7', 'ABCA8', 'HAPLN1', 'CCL8', 'DCN', 'EFEMP1', 'CFD', 'LUM', 'FBLN1', 'LINC01082', 'FN1', 'LMOD1', 'COL14A1', 'TDO2', 'SFRP1', 'SCARA5', 'ADH1B', 'NR2F1', 'TCF21', 'MFAP4', 'TNXB', 'THY1', 'EDIL3', 'BGN', 'CCDC80', 'HTRA3', 'IGFBP5', 'COL1A2', 'PDGFRB', 'GPC6', 'STMN2', 'WNT2B', 'IL34', 'MEG3', 'CFH', 'COL1A1', 'GGT5', 'TAGLN', 'MYLK', 'COL3A1', 'NID1', 'LOXL1', 'PTGDS', 'ITIH5', 'PDLIM3', 'COL5A2']
Epithelial = ['GUCA2B', 'AQP8', 'GUCA2A', 'LYPD8', 'PRAP1', 'MS4A12', 'CA1', 'SLC26A3', 'FABP1', 'ZG16', 'KRT20', 'HMGCS2', 'URAD', 'AKR1B10', 'CKMT1B', 'PHGR1', 'CHP2', 'CKMT1A', 'C1orf210', 'CEACAM7', 'TSPAN1', 'PPP1R14D', 'LGALS4', 'FABP2', 'MISP', 'SHD', 'C10orf99', 'PLA2G10', 'PRSS3', 'FXYD3', 'PRR15L', 'S100A14', 'PCK1', 'CDX2', 'PRR15', 'KRT8', 'EPCAM', 'CDHR5', 'KRT19', 'PRSS8', 'SFN', 'SMIM22', 'TFF1', 'CLDN7', 'CLDN4', 'GJB1', 'SLC39A5', 'USH1C', 'CBLC', 'TFF3']
T= ['CD3E', 'CD8A', 'CD3D', 'CD8B', 'TRBC1', 'CD3G', 'SH2D2A', 'GZMA', 'CD96', 'LAT', 'CD2', 'CCL5', 'TIGIT', 'CD7', 'ITK', 'CD6', 'GPR171', 'SPOCK2', 'ZAP70', 'MT-ATP8', 'CAMK4', 'GATA3', 'IL7R', 'OXNAD1', 'CD28', 'SYTL3', 'LCK', 'GZMM', 'BCL11B', 'ICOS', 'PRKCH', 'CD247', 'IFITM1', 'PTPN22', 'SH2D1A', 'FLT3LG', 'STAT4', 'CST7', 'TRAC', 'PDE4D', 'IL2RB', 'FYN', 'RORA', 'CBLB', 'TRAT1', 'SARAF', 'SKAP1', 'RGL4', 'CLEC2D', 'BATF']
Granulocyte= ['TPSAB1', 'TPSB2', 'CTSG', 'CMA1', 'CPA3', 'MS4A2', 'KRT1', 'HDC', 'SLC18A2', 'HPGDS', 'GATA2', 'KIT', 'MAOB', 'IL1RL1', 'LTC4S', 'SLC45A3', 'VWA5A', 'RGS13', 'RAB27B', 'FCER1A', 'MITF', 'HS3ST1', 'HPGD', 'STXBP6', 'ADRB2', 'CLU', 'CNRIP1', 'LEO1', 'ACOT7', 'ALOX5', 'CSF1', 'STX3', 'BACE2', 'BTK', 'GLUL', 'CAPG', 'DUSP6', 'FCER1G', 'ARHGAP18', 'PLIN2', 'ACSL4', 'ANXA1', 'ARHGEF6', 'STMN1', 'CD84', 'CTNNBL1', 'ALOX5AP', 'AREG', 'SAMSN1', 'RAB32']
Neuron = ['PLP1', 'LRRTM1', 'SOX10', 'NRXN1', 'MYOT', 'PCSK2', 'FOXD3', 'CDH19', 'CADM2', 'PTPRZ1', 'L1CAM', 'GFRA3', 'MPZ', 'COL28A1', 'TFAP2A', 'CRYAB', 'S100A1', 'XKR4', 'SORCS1', 'HAND2', 'TUBB2B', 'NTM', 'CDH2', 'TTR', 'NRXN3', 'NGFR', 'LGI4', 'MIA', 'PRIMA1', 'SCN7A', 'CADM4', 'TTYH1', 'S100B', 'NRN1', 'SPP1', 'ST6GALNAC2', 'KCNMB4', 'CADM3', 'GPM6B', 'NLGN4X', 'SEMA3B', 'ALDH1A1', 'TIMP4', 'TSPAN11', 'CLU', 'OLFML2A', 'ASPA', 'SNCA', 'COL8A1', 'SMIM5']
NK = ['NKG7', 'KLRB1', 'CD247', 'GZMM', 'TRBC2', 'TRAC', 'CTSW', 'CD7', 'GIMAP7', 'LCK', 'CCL5', 'CST7', 'CD2', 'IL7R', 'FYN', 'HCST', 'GZMA', 'RORA', 'SPOCK2', 'GIMAP4', 'EVL', 'CD96', 'CD3G', 'IL32', 'ACAP1', 'LEPROTL1', 'ARL4C', 'DUSP2', 'PTGER4', 'CCL4', 'ITM2A', 'TRAF3IP3', 'CREM', 'SOCS1', 'PTPN7', 'RGCC', 'PPP2R5C', 'TBC1D10C', 'ID2', 'CCND3', 'PCED1B-AS1', 'PTPRCAP', 'LTB', 'TNFAIP3', 'CD3D', 'EML4', 'ITGB2', 'CXCR4', 'CORO1A', 'ARHGEF1']
B= ['TCL1A', 'MS4A1', 'FCER2', 'VPREB3', 'BANK1', 'CD22', 'LINC00926', 'CD72', 'BCL11A', 'CD19', 'TNFRSF13C', 'IGHD', 'MARCH1', 'BLK', 'LY86', 'FCRLA', 'CXCR5', 'ARHGAP24', 'HVCN1', 'CD79B', 'SPIB', 'HLA-DOB', 'IRF8', 'CD83', 'P2RX5', 'SWAP70', 'LYL1', 'RALGPS2', 'HLA-DQB1', 'HLA-DMB', 'POU2F2', 'CD74', 'HLA-DQA2', 'GPR18', 'SMIM14', 'HLA-DQA1', 'LAT2', 'BCAS4', 'HLA-DRA', 'BLNK', 'CD40', 'MEF2C', 'LY9', 'CD37', 'CYB561A3', 'HLA-DPB1', 'HLA-DMA', 'HLA-DRB5', 'GGA2', 'HLA-DRB1']
Plasma = ['IGLC3', 'JCHAIN', 'IGHG2', 'IGLC2', 'IGHA2', 'IGHG1', 'IGHA1', 'IGKC', 'IGLV3-1', 'IGHG3', 'IGLC7', 'DERL3', 'IGHG4', 'MZB1', 'TNFRSF17', 'IGKV4-1', 'IGLV6-57', 'JSRP1', 'CCR10', 'IGHM', 'SPAG4', 'FKBP11', 'SSR4', 'FCRL5', 'SEC11C', 'SDC1', 'XBP1', 'CHPF', 'EAF2', 'PRDX4', 'SPINK2', 'PNOC', 'HERPUD1', 'PDK1', 'UBE2J1', 'POU2AF1', 'MEI1', 'MANEA', 'GNG7', 'CRELD2', 'SDF2L1', 'FKBP2', 'RAB30', 'ZBP1', 'SSR3', 'TXNDC11', 'SPATS2', 'DUSP5', 'MANF', 'CD38']
Endothelial =['TM4SF18', 'SOX18', 'NOVA2', 'SOX17', 'KDR', 'MYCT1', 'PLVAP', 'FAM110D', 'CDH5', 'MMRN2', 'CYYR1', 'FCN3', 'CLDN5', 'VWF', 'ECSCR', 'LCN6', 'RAMP3', 'ADGRL4', 'CD34', 'PCAT19', 'SHANK3', 'APLNR', 'NPR1', 'ZNF385D', 'BCL6B', 'PALMD', 'ROBO4', 'CLEC14A', 'SELP', 'EXOC3L2', 'ERG', 'PODXL', 'EGFL7', 'FAM167B', 'ARHGEF15', 'RAMP2', 'MADCAM1', 'PCDH17', 'CCL14', 'TEK', 'CALCRL', 'THSD7A', 'FLT1', 'SNCG', 'HSPA12B', 'JAM2', 'PTPRB', 'SLCO2A1', 'EMCN', 'TIE1']


In [ ]:
[i for i in Myeloid if i in adata.var_names.tolist()]

In [ ]:
[i for i in Connective if i in adata.var_names.tolist()]

In [ ]:
[i for i in Epithelial if i in adata.var_names.tolist()]

In [ ]:
[i for i in T if i in adata.var_names.tolist()]

In [ ]:
[i for i in Plasma if i in adata.var_names.tolist()]

In [ ]:
[i for i in Plasma if i in wtx.var_names.tolist()]

In [ ]:
markers = { 'Connective': [ 'COL1A1', 'TAGLN', 'COL3A1','IGFBP5'], 
            'Epithelial': [ 'KRT8', 'EPCAM', 'KRT19', 'CLDN4'],
           'Myeloid': [ 'LYZ','C1QB','C1QA', 'MS4A6A', 'CSF1R'], 
                     'Plasma': ['JCHAIN','IGHG1','IGKC'],
           'T': ['CD3E','CD8B','CCL5','SPOCK2','CD28']}

In [ ]:
adata_key = adata[adata.obs['cell_category_from_type'].isin(['T','Myeloid','Epithelial','Connective','Plasma'])]
dp = sc.pl.dotplot(
    adata_key, markers, 'cell_category_from_type', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()


In [ ]:
adata_key = adata[adata.obs['v7_ywl_b_imp_cell_category_from_type'].isin(['T','Myeloid','Epithelial','Connective','Plasma'])]
dp = sc.pl.dotplot(
    adata_key, markers, 'v7_ywl_b_imp_cell_category_from_type', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()


In [ ]:
wtx_key = wtx_inf[wtx_inf.obs['aucell_cell_category_from_type'].isin(['T','Myeloid','Epithelial','Connective','Plasma'])]
dp = sc.pl.dotplot(
    wtx_key, markers, 'aucell_cell_category_from_type', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()
